In [1]:


# Device Setup

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


KeyboardInterrupt: 


## Load / Prepare Data

Provide your raw text files and combine them into a single string.


In [ ]:


# Load Data 


texts = []
for fname in [
    "data/TheIdiot.txt",
    "data/theMetamorphosis.txt",
    "data/TheRepublic.txt",
    "data/shakespeareSonnets.txt",
    "data/williamBlakePoems.txt",
]:
    try:
        with open(fname, "r", encoding="utf-8") as f:
            texts.append(f.read())
    except FileNotFoundError:
        pass

text = "\n".join(texts)
print("Total characters:", len(text))


Total characters: 2796915



## Tokenizer (Character-Level)


In [ ]:


# Tokenizer

chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocab size:", vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

pad_id = stoi[chars[0]]  # simple pad token

data = encode(text)

# convert to torch tensor on the correct device
data_tensor = torch.tensor(data, dtype=torch.long, device=device)
# 90% train, 10% validation 
split = int(0.9 * data_tensor.size(0))
train_data = data_tensor[:split]
val_data = data_tensor[split:]
print('Train tokens:', train_data.size(0), 'Val tokens:', val_data.size(0))


Vocab size: 119
Train tokens: 2517223 Val tokens: 279692



## Model Components


In [ ]:

# Multi-Head Causal Self-Attention

class CausalSelfAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout=0.1):
        super().__init__()
        assert embedding_dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)

        # causal mask
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(context_length, context_length))
        )

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.size()  # Fix: Unpack the correct dimensions (B, T, D)

        qkv = self.qkv(x)  # (B, T, 3D)
        q, k, v = qkv.chunk(3, dim=-1)

        # reshape for multi-head
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # attention scores
        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        # weighted sum
        out = att @ v  # (B, H, T, Dh)

        # merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.proj(out)
        out = self.dropout(out)

        return out



In [ ]:


# Transformer Block

class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, context_length, num_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.attn = CausalSelfAttention(
            embedding_dim, num_heads, context_length, dropout
        )
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout1(self.attn(self.ln1(x)))
        x = x + self.dropout2(self.mlp(self.ln2(x)))
        return x


In [ ]:

# Token + Positional Embedding

class TokenPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, context_length, embedding_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embedding_dim)
        self.pos_emb = nn.Embedding(context_length, embedding_dim)

    def forward(self, x):
        # x: (B, T)
        B, T = x.shape

        positions = torch.arange(
            T,
            device=x.device
        ).unsqueeze(0)          # (1, T)

        tok = self.token_emb(x)             # (B, T, D)
        pos = self.pos_emb(positions)       # (1, T, D)

        return tok + pos                    # (B, T, D)


In [ ]:


# MiniGPT Model

class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim,
        num_blocks=3,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()
        self.embed = TokenPositionalEmbedding(
            vocab_size, context_length, embedding_dim
        )
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim, context_length, num_heads, dropout
                )
                for _ in range(num_blocks)
            ]
        )
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.head(x)



## Training


In [ ]:


# Hyperparameters

context_length = 256
embedding_dim = 128
num_blocks = 6
num_heads = 4
dropout = 0.1

model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_blocks=num_blocks,
    num_heads=num_heads,
    dropout=dropout
).to(device)

print("Model device:", next(model.parameters()).device)


Model device: cuda:0


In [ ]:
x_test = torch.randint(
    0, vocab_size,
    (32, context_length),
    device=device
)

out = model(x_test)
print(out.shape)


torch.Size([32, 256, 119])


In [ ]:

# Loss + Optimizer

def compute_loss(logits, targets):
    return F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1)
    )

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)



@torch.no_grad()
def estimate_loss(model, data, batch_size, context_length, steps=50):
    model.eval()
    losses = []

    for _ in range(steps):
        ix = torch.randint(
            0, data.size(0) - context_length - 1,
            (batch_size,),
            device=data.device
        )

        x = torch.stack([data[i:i+context_length] for i in ix])
        y = torch.stack([data[i+1:i+context_length+1] for i in ix])

        logits = model(x)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)



In [ ]:

# Training Loop 
num_steps = 5000
print_every = 500
batch_size = 64

for step in range(num_steps):

    # sample batch of starting indices (on GPU)
    ix = torch.randint(
        0,
        len(data) - context_length - 1,
        (batch_size,),
        device=device
        )

    # build batch (vectorized, fast)
    x = torch.stack([
        torch.tensor(
            data[i : i + context_length],
            dtype=torch.long,
            device=device
        )
        for i in ix
    ])

    y = torch.stack([
        torch.tensor(
            data[i + 1 : i + context_length + 1],
            dtype=torch.long,
            device=device
        )
        for i in ix
    ])

    # forward
    logits = model(x)        # (B, T, V)

    # loss
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        y.view(-1)
    )

    # backward
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % print_every == 0:
        print(f"step {step} | loss {loss.item():.4f}")

train_loss = loss.item()
val_loss = estimate_loss(
    model,
    val_data,
    batch_size,
    context_length
)

print(f"train {train_loss:.4f} | val {val_loss:.4f}")


step 0 | loss 5.0124
step 500 | loss 2.4531
step 1000 | loss 2.1873
step 1500 | loss 1.9791
step 2000 | loss 1.8173
step 2500 | loss 1.7801
step 3000 | loss 1.6758
step 3500 | loss 1.6640
step 4000 | loss 1.6762
step 4500 | loss 1.6428
train 1.6552 | val 1.6513



## Text Generation


In [ ]:

@torch.no_grad()
def generate(model, start_text, length, temperature=1.0):
    model.eval()

    encoded = encode(start_text)

    if len(encoded) < context_length:
        encoded = [pad_id] * (context_length - len(encoded)) + encoded
    else:
        encoded = encoded[-context_length:]

    output = encoded.copy()

    for _ in range(length):
        x = torch.tensor(
            output[-context_length:],
            dtype=torch.long,
            device=device
        ).unsqueeze(0)  # (1, T)

        logits = model(x)           # (1, T, V)
        logits = logits[0, -1] / temperature  # (V,) last token
        probs = F.softmax(logits, dim=-1)

        next_char = torch.multinomial(probs, 1).item()
        output.append(next_char)

    return decode(output)


In [ ]:

print(generate(model, "the rain was pouring", 400, temperature=0.9))




















































































































































































































































Gregor Samsa 

  Be  LSo     He  "


Bund           Fon      ‘Andidesmis that ance and the go,
   And if thead at litto be him?





—ANIThe wlearit,
      Tyread delacter of beet my mean telle,
    Yearthatly the earther with booth;
     Ohe we not is a which who real of the vain,
      Simply, but after not in there propened with a list I
seally thought it the most in sol what luttled in it mine
this aguate 


SyntaxError: invalid syntax (476313318.py, line 1)